# Interseção SICOR Cerrado × imóveis do SICAR com cadastro suspenso por ID_GLEBA2

Este notebook cruza o arquivo `sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv` com 13 shapefiles estaduais do SICAR localizados na pasta `Base/CAR`.

## Regra do impedimento

- somente entram no cruzamento as feições cujo campo `ind_status`, após retirar espaços e padronizar maiúsculas/minúsculas, seja igual a **Suspenso**;
- os demais estados cadastrais são contabilizados apenas no arquivo de diagnóstico e não são tratados como impedimento;
- cada `ID_GLEBA2` distinto é submetido à interseção espacial somente uma vez;
- quando o mesmo `ID_GLEBA2` aparece em várias linhas, o resultado calculado é reaproveitado;
- antes do processamento, o notebook valida se um mesmo `ID_GLEBA2` está sempre associado à mesma geometria;
- a deduplicação técnica usa a chave `ID_GLEBA2 + CAR_ID_INTERNO`, e não `REF_BACEN`.

## Saídas

1. **Resultado espacial único:** uma linha por `ID_GLEBA2 × imóvel do SICAR suspenso`.
2. **Resultado detalhado expandido:** associa a resposta espacial a todas as linhas originais que possuem aquele `ID_GLEBA2`.
3. **Resumo por ID_GLEBA2:** contém todos os IDs, inclusive os que não intersectaram imóveis suspensos.
4. **Diagnóstico de status:** contagem dos valores de `ind_status` por UF antes do filtro.

Os atributos dos shapefiles são preservados com o prefixo `CAR_`, inclusive `CAR_IND_STATUS`, a UF e o arquivo de origem.


In [1]:
# ============================================================
# 1. INSTALAR, IMPORTAR E MONTAR O GOOGLE DRIVE
# ============================================================

%pip install -q --upgrade geopandas pyogrio shapely

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio

from shapely import force_2d, intersection as shapely_intersection, make_valid, set_precision
from shapely.errors import GEOSException

warnings.filterwarnings('ignore', category=UserWarning)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 180)

print('Ambiente preparado.')


Mounted at /content/drive
Ambiente preparado.


In [ ]:
# ============================================================
# 2. CAMINHOS E PARÂMETROS
# ============================================================

ARQUIVO_SICOR = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/Base04_enriquecimento_detalhado/"
    "sicor_cerrado_2021_2025_enriquecido_detalhado_com_id_gleba2.csv"
)

PASTA_SICAR = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/Base/CAR"
)

UFS_SICAR = [
    'ac', 'ba', 'df', 'go', 'ma', 'mg', 'ms',
    'mt', 'pi', 'pr', 'ro', 'sp', 'to',
]

ARQUIVOS_SICAR = {
    uf.upper(): PASTA_SICAR / f'sicar_imoveis_{uf}.shp'
    for uf in UFS_SICAR
}

# Pasta exclusiva para não sobrescrever resultados de outros impedimentos.
PASTA_SAIDA = Path(
    "/content/drive/MyDrive/Colab Notebooks/fluxo/Envio/"
    "Arquivos gerados e relatorios/intersecoes/sicar_suspensos/"
    "resultados_por_id_gleba2"
)

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVO_INTERSECAO_POR_ID = (
    PASTA_SAIDA / "intersecao_sicar_suspensos_unica_por_id_gleba2.csv"
)

ARQUIVO_INTERSECAO_EXPANDIDA = (
    PASTA_SAIDA / "intersecao_sicor_sicar_suspensos_expandida_por_id_gleba2.csv"
)

ARQUIVO_RESUMO_POR_ID = (
    PASTA_SAIDA / "resumo_sicar_suspensos_por_id_gleba2.csv"
)

ARQUIVO_DIAGNOSTICO_STATUS = (
    PASTA_SAIDA / "diagnostico_ind_status_sicar_por_uf.csv"
)

CRS_SICOR = "EPSG:4326"
CRS_AREA = "EPSG:5880"
TAMANHO_BLOCO = 20_000
SEPARADOR_SAIDA = ";"
STATUS_IMPEDIMENTO = "SUSPENSO"

print(f"{'OK' if ARQUIVO_SICOR.exists() else 'NÃO ENCONTRADO'} | {ARQUIVO_SICOR}")

arquivos_ausentes = []
for uf, caminho in ARQUIVOS_SICAR.items():
    existe = caminho.exists()
    print(f"{'OK' if existe else 'NÃO ENCONTRADO'} | {uf} | {caminho.name}")
    if not existe:
        arquivos_ausentes.append(caminho)

if not ARQUIVO_SICOR.exists():
    raise FileNotFoundError(f"CSV do SICOR não encontrado: {ARQUIVO_SICOR}")

if arquivos_ausentes:
    raise FileNotFoundError(
        "Shapefiles do SICAR não encontrados:\n- "
        + "\n- ".join(str(caminho) for caminho in arquivos_ausentes)
    )

for caminho in ARQUIVOS_SICAR.values():
    for extensao in ['.dbf', '.shx', '.prj']:
        componente = caminho.with_suffix(extensao)
        if not componente.exists():
            print(f"ATENÇÃO | componente ausente: {componente}")

print("\nPasta de saída:")
print(PASTA_SAIDA)


In [ ]:
# ============================================================
# 3. FUNÇÕES AUXILIARES
# ============================================================

CODIFICACOES = ['utf-8-sig', 'utf-8', 'latin-1', 'cp1252']
SEPARADORES = [';', ',', '	', '|']


def remover_acentos(texto):
    return ''.join(
        caractere
        for caractere in unicodedata.normalize('NFKD', str(texto))
        if not unicodedata.combining(caractere)
    )


def normalizar_nome_coluna(nome):
    nome = remover_acentos(nome).upper().strip().replace('#', '')
    nome = re.sub(r'[^A-Z0-9]+', '_', nome)
    return re.sub(r'_+', '_', nome).strip('_')


def normalizar_colunas(df):
    df = df.copy()
    df.columns = [normalizar_nome_coluna(coluna) for coluna in df.columns]

    if df.columns.duplicated().any():
        duplicadas = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f'A normalização produziu colunas duplicadas: {duplicadas}')

    return df


def normalizar_valor_texto(serie):
    return (
        serie.astype('string')
        .fillna('')
        .map(remover_acentos)
        .str.upper()
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )


def detectar_csv(caminho):
    ultimo_erro = None

    for encoding in CODIFICACOES:
        for separador in SEPARADORES:
            try:
                teste = pd.read_csv(
                    caminho,
                    sep=separador,
                    encoding=encoding,
                    nrows=5,
                    dtype=str,
                    low_memory=False,
                )

                if teste.shape[1] <= 1:
                    continue

                teste = normalizar_colunas(teste)
                return encoding, separador, list(teste.columns)

            except Exception as erro:
                ultimo_erro = erro

    raise RuntimeError(
        'Não foi possível detectar a estrutura do CSV. '
        f'Último erro: {ultimo_erro}'
    )


def localizar_coluna(colunas, candidatos, obrigatoria=True):
    mapa_colunas = {
        normalizar_nome_coluna(coluna): coluna
        for coluna in colunas
    }

    for candidato in [normalizar_nome_coluna(c) for c in candidatos]:
        if candidato in mapa_colunas:
            return mapa_colunas[candidato]

    if obrigatoria:
        raise KeyError(
            'Nenhuma das colunas esperadas foi encontrada: '
            f'{[normalizar_nome_coluna(c) for c in candidatos]}'
        )

    return None


def padronizar_codigo(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.lstrip('﻿')
        .str.replace(r'\.0$', '', regex=True)
        .replace({'': pd.NA, 'NAN': pd.NA, 'NONE': pd.NA, '<NA>': pd.NA})
    )


def corrigir_geometrias(gdf):
    quantidade_entrada = len(gdf)

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    mascara_invalida = ~gdf.geometry.is_valid
    quantidade_invalidas = int(mascara_invalida.sum())

    if quantidade_invalidas:
        try:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].make_valid()
            )
        except Exception:
            gdf.loc[mascara_invalida, 'geometry'] = (
                gdf.loc[mascara_invalida, 'geometry'].buffer(0)
            )

    gdf = gdf[
        gdf.geometry.notna()
        & ~gdf.geometry.is_empty
        & gdf.geometry.is_valid
    ].copy()

    quantidade_descartadas = quantidade_entrada - len(gdf)
    return gdf, quantidade_invalidas, quantidade_descartadas


def serie_wkt_para_geometria(serie):
    return gpd.GeoSeries.from_wkt(
        serie.astype('string'),
        crs=CRS_SICOR,
        on_invalid='ignore',
    )


def preparar_geometria_topologica(geometria, precisao=None):
    if geometria is None:
        return None

    try:
        if geometria.is_empty:
            return None
        geometria = force_2d(geometria)

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        if precisao is not None:
            geometria = set_precision(
                geometria,
                grid_size=precisao,
                mode='valid_output',
            )

        if geometria is None or geometria.is_empty:
            return None

        if not geometria.is_valid:
            geometria = make_valid(geometria)

        return geometria

    except Exception:
        try:
            geometria = geometria.buffer(0)
            return None if geometria.is_empty else geometria
        except Exception:
            return None


def calcular_intersecoes_seguras(geometrias_esquerda, geometrias_direita, crs):
    resultados = []
    quantidade_recuperadas = 0
    quantidade_falhas = 0

    for geometria_esquerda, geometria_direita in zip(
        geometrias_esquerda.to_numpy(),
        geometrias_direita.to_numpy(),
    ):
        esquerda = preparar_geometria_topologica(geometria_esquerda, 0.01)
        direita = preparar_geometria_topologica(geometria_direita, 0.01)

        if esquerda is None or direita is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        intersecao = None
        recuperada = False

        try:
            intersecao = shapely_intersection(esquerda, direita, grid_size=0.01)
        except (GEOSException, ValueError, TypeError):
            recuperada = True

            try:
                esquerda_10 = preparar_geometria_topologica(esquerda, 0.10)
                direita_10 = preparar_geometria_topologica(direita, 0.10)

                if esquerda_10 is not None and direita_10 is not None:
                    intersecao = shapely_intersection(
                        esquerda_10,
                        direita_10,
                        grid_size=0.10,
                    )
            except (GEOSException, ValueError, TypeError):
                intersecao = None

            if intersecao is None:
                try:
                    intersecao = esquerda.buffer(0).intersection(direita.buffer(0))
                except Exception:
                    intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
            continue

        try:
            if not intersecao.is_empty and not intersecao.is_valid:
                intersecao = make_valid(intersecao)
        except Exception:
            try:
                intersecao = intersecao.buffer(0)
            except Exception:
                intersecao = None

        if intersecao is None:
            resultados.append(None)
            quantidade_falhas += 1
        else:
            resultados.append(intersecao)
            if recuperada:
                quantidade_recuperadas += 1

    return (
        gpd.GeoSeries(resultados, index=pd.RangeIndex(len(resultados)), crs=crs),
        quantidade_recuperadas,
        quantidade_falhas,
    )


In [ ]:
# ============================================================
# 4. LER, FILTRAR E CONSOLIDAR OS SHAPEFILES DO SICAR
# ============================================================

camadas_suspensas = []
diagnosticos_status = []
total_feicoes_lidas = 0
total_feicoes_suspensas = 0
total_invalidas_corrigidas = 0
total_descartadas = 0

for uf, caminho in ARQUIVOS_SICAR.items():
    informacoes = pyogrio.read_info(caminho)
    campos_originais = [str(campo) for campo in informacoes['fields']]
    coluna_status_original = localizar_coluna(
        campos_originais,
        ['ind_status'],
    )

    # Lê apenas a coluna de status para produzir o diagnóstico sem carregar
    # todas as geometrias do estado na memória.
    tabela_status = pyogrio.read_dataframe(
        caminho,
        columns=[coluna_status_original],
        read_geometry=False,
    )
    total_feicoes_lidas += len(tabela_status)

    status_original = tabela_status[coluna_status_original].astype('string')
    status_normalizado = normalizar_valor_texto(status_original)

    diagnostico_uf = (
        pd.DataFrame({
            'UF': uf,
            'IND_STATUS_ORIGINAL': status_original.fillna('<NULO>').str.strip(),
            'IND_STATUS_NORMALIZADO': status_normalizado.replace({'': '<NULO>'}),
        })
        .value_counts(dropna=False)
        .rename('QUANTIDADE')
        .reset_index()
    )
    diagnosticos_status.append(diagnostico_uf)

    quantidade_suspensa_esperada = int(status_normalizado.eq(STATUS_IMPEDIMENTO).sum())
    del tabela_status, status_original, status_normalizado
    gc.collect()

    # O filtro na leitura evita carregar milhões de geometrias que não são
    # impedimento. Se o driver não aceitar a expressão, há fallback seguro.
    expressao_filtro = (
        f"UPPER(TRIM(\"{coluna_status_original}\")) = "
        f"'{STATUS_IMPEDIMENTO}'"
    )

    try:
        camada = pyogrio.read_dataframe(
            caminho,
            where=expressao_filtro,
        )
    except Exception as erro_filtro:
        print(f'{uf}: filtro direto indisponível; usando fallback em memória.')
        print('Motivo:', erro_filtro)
        camada = pyogrio.read_dataframe(caminho)

    crs_original = camada.crs
    if crs_original is None:
        raise ValueError(
            f'O shapefile {caminho.name} não possui CRS. '
            'Verifique se o arquivo .prj está na mesma pasta.'
        )

    camada = normalizar_colunas(camada)
    coluna_status = localizar_coluna(camada.columns, ['ind_status'])
    coluna_geometria = localizar_coluna(camada.columns, ['geometry'])

    camada = gpd.GeoDataFrame(
        camada,
        geometry=coluna_geometria,
        crs=crs_original,
    )
    if camada.geometry.name != 'geometry':
        camada = camada.rename_geometry('geometry')

    # Reaplica o filtro após a leitura: esta é a garantia final de que
    # nenhum status diferente de SUSPENSO será tratado como impedimento.
    camada = camada[
        normalizar_valor_texto(camada[coluna_status]).eq(STATUS_IMPEDIMENTO)
    ].copy()

    if len(camada) != quantidade_suspensa_esperada:
        print(
            f'ATENÇÃO | {uf}: diagnóstico={quantidade_suspensa_esperada:,} | '
            f'geometrias filtradas={len(camada):,}'
        )

    if camada.empty:
        print(f'{uf}: nenhuma feição com ind_status = Suspenso.')
        continue

    camada['UF_ORIGEM'] = uf
    camada['ARQUIVO_ORIGEM'] = caminho.name
    camada, invalidas, descartadas = corrigir_geometrias(camada)
    camada = camada.to_crs(CRS_SICOR).reset_index(drop=True)

    total_feicoes_suspensas += len(camada)
    total_invalidas_corrigidas += invalidas
    total_descartadas += descartadas
    camadas_suspensas.append(camada)

    print(
        f'{uf}: suspensas válidas={len(camada):,} | '
        f'inválidas corrigidas={invalidas:,} | descartadas={descartadas:,}'
    )

diagnostico_status = pd.concat(
    diagnosticos_status,
    ignore_index=True,
    sort=False,
)
diagnostico_status.to_csv(
    ARQUIVO_DIAGNOSTICO_STATUS,
    index=False,
    sep=SEPARADOR_SAIDA,
    encoding='utf-8-sig',
)

if not camadas_suspensas:
    raise RuntimeError(
        'Nenhuma feição com ind_status = Suspenso foi localizada nos 13 shapefiles.'
    )

sicar_suspenso = gpd.GeoDataFrame(
    pd.concat(camadas_suspensas, ignore_index=True, sort=False),
    geometry='geometry',
    crs=CRS_SICOR,
)
del camadas_suspensas, diagnosticos_status
gc.collect()

sicar_suspenso['CAR_ID_INTERNO'] = np.arange(1, len(sicar_suspenso) + 1)

renomear_car = {
    coluna: f'CAR_{coluna}'
    for coluna in sicar_suspenso.columns
    if coluna not in {'geometry', 'CAR_ID_INTERNO'}
}
sicar_suspenso = sicar_suspenso.rename(columns=renomear_car).set_geometry('geometry')

sicar_area = sicar_suspenso.to_crs(CRS_AREA)
sicar_suspenso['CAR_AREA_TOTAL_HA'] = sicar_area.geometry.area / 10_000

COLUNAS_CAR_SAIDA = [
    coluna for coluna in sicar_suspenso.columns if coluna != 'geometry'
]

if 'CAR_IND_STATUS' not in sicar_suspenso.columns:
    raise KeyError('A coluna CAR_IND_STATUS não foi preservada após a consolidação.')

status_final = normalizar_valor_texto(sicar_suspenso['CAR_IND_STATUS'])
if not status_final.eq(STATUS_IMPEDIMENTO).all():
    raise RuntimeError('A camada consolidada contém status diferente de SUSPENSO.')

print('\nCONSOLIDAÇÃO DO SICAR CONCLUÍDA')
print('=' * 90)
print('Feições examinadas no diagnóstico:', f'{total_feicoes_lidas:,}')
print('Feições suspensas válidas:', f'{total_feicoes_suspensas:,}')
print('Geometrias inválidas corrigidas:', f'{total_invalidas_corrigidas:,}')
print('Geometrias descartadas:', f'{total_descartadas:,}')
print('CRS espacial:', sicar_suspenso.crs)
print('CRS para áreas:', CRS_AREA)
print('Diagnóstico salvo em:', ARQUIVO_DIAGNOSTICO_STATUS)

display(
    sicar_suspenso[
        ['CAR_ID_INTERNO', 'CAR_UF_ORIGEM', 'CAR_IND_STATUS', 'CAR_AREA_TOTAL_HA']
    ].head()
)


In [ ]:
# ============================================================
# 5. DETECTAR E VALIDAR AS COLUNAS DO SICOR
# ============================================================

encoding_sicor, separador_sicor, colunas_sicor = detectar_csv(ARQUIVO_SICOR)

COLUNA_ID_GLEBA2 = localizar_coluna(
    colunas_sicor,
    ['ID_GLEBA2', 'ID_GLEBA_2'],
)

COLUNA_GEOMETRIA_SICOR = localizar_coluna(
    colunas_sicor,
    ['GT_GEOMETRIA', 'GT_GEOMETRIA_REPRESENTATIVA', 'GEOMETRIA', 'GEOMETRY', 'WKT'],
)

COLUNA_REF_BACEN = localizar_coluna(
    colunas_sicor,
    ['REF_BACEN', '#REF_BACEN', 'CD_REF_BACEN'],
    obrigatoria=False,
)

print('Codificação:', encoding_sicor)
print('Separador:', repr(separador_sicor))
print('Coluna-chave:', COLUNA_ID_GLEBA2)
print('Coluna de geometria:', COLUNA_GEOMETRIA_SICOR)
print('Coluna REF_BACEN:', COLUNA_REF_BACEN)
print('Quantidade de colunas:', len(colunas_sicor))

if COLUNA_ID_GLEBA2 == COLUNA_GEOMETRIA_SICOR:
    raise RuntimeError('A coluna do ID e a coluna geométrica não podem ser a mesma.')


In [ ]:
# ============================================================
# 6. INTERSECTAR UMA ÚNICA VEZ CADA ID_GLEBA2
# ============================================================

# Guarda somente uma assinatura numérica da geometria de cada ID.
# Isso permite reconhecer IDs já processados sem manter todos os WKT na memória.
assinatura_geometria_por_id = {}
resultados_intersecao = []

total_linhas_lidas = 0
total_linhas_com_id_e_geometria = 0
total_ids_novos = 0
total_repeticoes_puladas = 0
total_ids_geometria_valida = 0
total_geometrias_invalidas_corrigidas = 0
total_geometrias_descartadas = 0
total_candidatos_espaciais = 0
total_intersecoes_validas = 0
total_intersecoes_recuperadas = 0
total_falhas_topologicas = 0
total_contatos_sem_area = 0

leitor = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor, start=1):
    bloco = normalizar_colunas(bloco)
    total_linhas_lidas += len(bloco)

    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    bloco[COLUNA_GEOMETRIA_SICOR] = bloco[COLUNA_GEOMETRIA_SICOR].astype('string').str.strip()

    bloco = bloco.dropna(
        subset=[COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
    ).copy()

    total_linhas_com_id_e_geometria += len(bloco)

    if bloco.empty:
        print(f'Bloco {numero_bloco:03d}: sem ID_GLEBA2 e geometria válidos')
        continue

    bloco['_HASH_GEOMETRIA'] = (
        pd.util.hash_pandas_object(
            bloco[COLUNA_GEOMETRIA_SICOR],
            index=False,
        )
        .astype('uint64')
    )

    # Validação dentro do próprio bloco.
    geometrias_por_id_no_bloco = bloco.groupby(
        COLUNA_ID_GLEBA2,
        dropna=False,
    )['_HASH_GEOMETRIA'].nunique()

    ids_inconsistentes = geometrias_por_id_no_bloco[
        geometrias_por_id_no_bloco > 1
    ].index.tolist()

    if ids_inconsistentes:
        raise ValueError(
            'Um mesmo ID_GLEBA2 foi associado a geometrias diferentes. '
            f'Primeiros IDs inconsistentes: {ids_inconsistentes[:10]}'
        )

    representantes = bloco.drop_duplicates(
        subset=[COLUNA_ID_GLEBA2],
        keep='first',
    ).reset_index(drop=True)

    mascara_id_novo = []

    # Validação também contra IDs que apareceram em blocos anteriores.
    for id_atual, hash_atual in representantes[
        [COLUNA_ID_GLEBA2, '_HASH_GEOMETRIA']
    ].itertuples(index=False, name=None):
        hash_atual = int(hash_atual)

        if id_atual in assinatura_geometria_por_id:
            if assinatura_geometria_por_id[id_atual] != hash_atual:
                raise ValueError(
                    f'O ID_GLEBA2 {id_atual} apareceu com geometrias diferentes '
                    'em blocos distintos do CSV.'
                )
            mascara_id_novo.append(False)
        else:
            assinatura_geometria_por_id[id_atual] = hash_atual
            mascara_id_novo.append(True)

    representantes_novos = representantes.loc[
        mascara_id_novo
    ].copy().reset_index(drop=True)

    total_ids_novos += len(representantes_novos)
    total_repeticoes_puladas += len(bloco) - len(representantes_novos)

    if representantes_novos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            'todos os IDs já haviam sido processados'
        )
        del bloco, representantes, representantes_novos
        gc.collect()
        continue

    geometria = serie_wkt_para_geometria(
        representantes_novos[COLUNA_GEOMETRIA_SICOR]
    )

    gdf_ids = gpd.GeoDataFrame(
        representantes_novos[
            [COLUNA_ID_GLEBA2, COLUNA_GEOMETRIA_SICOR]
        ].copy(),
        geometry=geometria,
        crs=CRS_SICOR,
    )

    gdf_ids, invalidas_corrigidas, descartadas = corrigir_geometrias(gdf_ids)
    gdf_ids = gdf_ids.reset_index(drop=True)

    total_geometrias_invalidas_corrigidas += invalidas_corrigidas
    total_geometrias_descartadas += descartadas
    total_ids_geometria_valida += len(gdf_ids)

    if gdf_ids.empty:
        print(
            f'Bloco {numero_bloco:03d}: IDs novos={len(representantes_novos):,} | '
            'nenhuma geometria válida'
        )
        del bloco, representantes, representantes_novos, gdf_ids
        gc.collect()
        continue

    candidatos = gpd.sjoin(
        gdf_ids,
        sicar_suspenso,
        how='inner',
        predicate='intersects',
    ).reset_index(drop=True)

    total_candidatos_espaciais += len(candidatos)

    if candidatos.empty:
        print(
            f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
            f'IDs novos={len(representantes_novos):,} | interseções=0 | '
            f'repetições puladas={len(bloco) - len(representantes_novos):,}'
        )
        del bloco, representantes, representantes_novos, gdf_ids, candidatos
        gc.collect()
        continue

    candidatos['GEOMETRIA_TI'] = (
        candidatos['index_right'].map(sicar_suspenso.geometry)
    )

    candidatos = candidatos.dropna(
        subset=['GEOMETRIA_TI']
    ).reset_index(drop=True)

    geometria_gleba_wgs84 = gpd.GeoSeries(
        candidatos.geometry.to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_car_wgs84 = gpd.GeoSeries(
        candidatos['GEOMETRIA_TI'].to_numpy(),
        index=pd.RangeIndex(len(candidatos)),
        crs=CRS_SICOR,
    )

    geometria_gleba_area = geometria_gleba_wgs84.to_crs(CRS_AREA)
    geometria_car_area = geometria_car_wgs84.to_crs(CRS_AREA)

    geometria_gleba_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_gleba_area],
        index=pd.RangeIndex(len(geometria_gleba_area)),
        crs=CRS_AREA,
    )

    geometria_car_area = gpd.GeoSeries(
        [preparar_geometria_topologica(g, 0.01) for g in geometria_car_area],
        index=pd.RangeIndex(len(geometria_car_area)),
        crs=CRS_AREA,
    )

    (
        geometria_intersecao_area,
        recuperadas_bloco,
        falhas_bloco,
    ) = calcular_intersecoes_seguras(
        geometria_gleba_area,
        geometria_car_area,
        CRS_AREA,
    )

    total_intersecoes_recuperadas += recuperadas_bloco
    total_falhas_topologicas += falhas_bloco

    candidatos['AREA_GLEBA_HA_CALCULADA'] = (
        geometria_gleba_area.area.to_numpy() / 10_000
    )
    candidatos['AREA_INTERSECAO_HA'] = (
        geometria_intersecao_area.area.to_numpy() / 10_000
    )
    candidatos['PERCENTUAL_GLEBA_INTERSECTADA'] = np.where(
        candidatos['AREA_GLEBA_HA_CALCULADA'] > 0,
        candidatos['AREA_INTERSECAO_HA']
        / candidatos['AREA_GLEBA_HA_CALCULADA']
        * 100,
        np.nan,
    )

    mascara_area_positiva = (
        candidatos['AREA_INTERSECAO_HA'].fillna(0).gt(0).to_numpy()
    )
    total_contatos_sem_area += int((~mascara_area_positiva).sum())

    if not mascara_area_positiva.any():
        print(
            f'Bloco {numero_bloco:03d}: candidatos={len(candidatos):,} | '
            'nenhuma interseção com área positiva'
        )
        del (
            bloco, representantes, representantes_novos, gdf_ids, candidatos,
            geometria_gleba_wgs84, geometria_car_wgs84,
            geometria_gleba_area, geometria_car_area,
            geometria_intersecao_area, mascara_area_positiva,
        )
        gc.collect()
        continue

    candidatos = candidatos.loc[mascara_area_positiva].reset_index(drop=True)

    geometria_intersecao_area_valida = gpd.GeoSeries(
        geometria_intersecao_area.to_numpy()[mascara_area_positiva],
        index=pd.RangeIndex(int(mascara_area_positiva.sum())),
        crs=CRS_AREA,
    )
    geometria_intersecao_wgs84 = geometria_intersecao_area_valida.to_crs(CRS_SICOR)

    candidatos['GEOMETRIA_CAR_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in candidatos['GEOMETRIA_TI']
    ]
    candidatos['GEOMETRIA_INTERSECAO_WKT'] = [
        g.wkt if g is not None and not g.is_empty else pd.NA
        for g in geometria_intersecao_wgs84
    ]
    candidatos['TIPO_IMPEDIMENTO'] = 'Cadastro Ambiental Rural (CAR)'
    candidatos['STATUS_IMPEDIMENTO'] = 'Suspenso'

    colunas_resultado = [
        COLUNA_ID_GLEBA2,
        *COLUNAS_CAR_SAIDA,
        'AREA_GLEBA_HA_CALCULADA',
        'AREA_INTERSECAO_HA',
        'PERCENTUAL_GLEBA_INTERSECTADA',
        'GEOMETRIA_CAR_WKT',
        'GEOMETRIA_INTERSECAO_WKT',
        'TIPO_IMPEDIMENTO',
        'STATUS_IMPEDIMENTO',
    ]

    resultado_bloco = pd.DataFrame(candidatos[colunas_resultado])
    resultados_intersecao.append(resultado_bloco)
    total_intersecoes_validas += len(resultado_bloco)

    print(
        f'Bloco {numero_bloco:03d}: lidas={len(bloco):,} | '
        f'IDs novos={len(representantes_novos):,} | '
        f'repetições puladas={len(bloco) - len(representantes_novos):,} | '
        f'candidatos={len(mascara_area_positiva):,} | '
        f'interseções com área={len(resultado_bloco):,}'
    )

    del (
        bloco, representantes, representantes_novos, gdf_ids, candidatos,
        geometria_gleba_wgs84, geometria_car_wgs84,
        geometria_gleba_area, geometria_car_area,
        geometria_intersecao_area, geometria_intersecao_area_valida,
        geometria_intersecao_wgs84, mascara_area_positiva, resultado_bloco,
    )
    gc.collect()

print('\nPROCESSAMENTO ESPACIAL CONCLUÍDO')
print('=' * 90)
print('Linhas lidas:', f'{total_linhas_lidas:,}')
print('Linhas com ID e geometria:', f'{total_linhas_com_id_e_geometria:,}')
print('ID_GLEBA2 distintos:', f'{len(assinatura_geometria_por_id):,}')
print('IDs submetidos ao teste:', f'{total_ids_novos:,}')
print('Repetições que não foram testadas novamente:', f'{total_repeticoes_puladas:,}')
print('IDs com geometria válida:', f'{total_ids_geometria_valida:,}')
print('Geometrias inválidas corrigidas:', f'{total_geometrias_invalidas_corrigidas:,}')
print('Geometrias descartadas:', f'{total_geometrias_descartadas:,}')
print('Candidatos espaciais:', f'{total_candidatos_espaciais:,}')
print('Interseções recuperadas:', f'{total_intersecoes_recuperadas:,}')
print('Falhas topológicas:', f'{total_falhas_topologicas:,}')
print('Contatos sem área positiva:', f'{total_contatos_sem_area:,}')
print('Interseções válidas ID_GLEBA2 × cadastro do SICAR suspenso:', f'{total_intersecoes_validas:,}')


In [ ]:
# ============================================================
# 7. CONSOLIDAR E SALVAR O RESULTADO ÚNICO POR ID_GLEBA2
# ============================================================

if resultados_intersecao:
    intersecoes_por_id = pd.concat(
        resultados_intersecao,
        ignore_index=True,
        sort=False,
    )
else:
    intersecoes_por_id = pd.DataFrame(
        columns=[COLUNA_ID_GLEBA2, 'CAR_ID_INTERNO']
    )

del resultados_intersecao
gc.collect()

quantidade_antes = len(intersecoes_por_id)

if not intersecoes_por_id.empty:
    intersecoes_por_id = (
        intersecoes_por_id
        .sort_values('AREA_INTERSECAO_HA', ascending=False)
        .drop_duplicates(
            subset=[COLUNA_ID_GLEBA2, 'CAR_ID_INTERNO'],
            keep='first',
        )
        .reset_index(drop=True)
    )

print(
    'Duplicações técnicas ID_GLEBA2 × cadastro do SICAR suspenso removidas:',
    f'{quantidade_antes - len(intersecoes_por_id):,}',
)

intersecoes_por_id.to_csv(
    ARQUIVO_INTERSECAO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

# O resumo inclui todos os IDs, inclusive os sem interseção.
todos_ids = pd.DataFrame({
    COLUNA_ID_GLEBA2: list(assinatura_geometria_por_id.keys())
})

if intersecoes_por_id.empty:
    resumo_por_id = todos_ids.copy()
    resumo_por_id['TEM_INTERSECAO_CAR_SUSPENSO'] = False
    resumo_por_id['QUANTIDADE_CADASTROS_CAR_SUSPENSOS_INTERSECTADOS'] = 0
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = 0.0
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = 0.0
else:
    resumo_positivo = (
        intersecoes_por_id.groupby(COLUNA_ID_GLEBA2, dropna=False)
        .agg(
            QUANTIDADE_CADASTROS_CAR_SUSPENSOS_INTERSECTADOS=(
                'CAR_ID_INTERNO', 'nunique'
            ),
            AREA_INTERSECAO_TOTAL_HA=('AREA_INTERSECAO_HA', 'sum'),
            PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA=(
                'PERCENTUAL_GLEBA_INTERSECTADA', 'max'
            ),
        )
        .reset_index()
    )

    resumo_por_id = todos_ids.merge(
        resumo_positivo,
        on=COLUNA_ID_GLEBA2,
        how='left',
        validate='one_to_one',
    )

    resumo_por_id['TEM_INTERSECAO_CAR_SUSPENSO'] = (
        resumo_por_id['QUANTIDADE_CADASTROS_CAR_SUSPENSOS_INTERSECTADOS']
        .notna()
    )
    resumo_por_id['QUANTIDADE_CADASTROS_CAR_SUSPENSOS_INTERSECTADOS'] = (
        resumo_por_id['QUANTIDADE_CADASTROS_CAR_SUSPENSOS_INTERSECTADOS']
        .fillna(0)
        .astype('int64')
    )
    resumo_por_id['AREA_INTERSECAO_TOTAL_HA'] = (
        resumo_por_id['AREA_INTERSECAO_TOTAL_HA'].fillna(0.0)
    )
    resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'] = (
        resumo_por_id['PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA'].fillna(0.0)
    )

    resumo_por_id = resumo_por_id[
        [
            COLUNA_ID_GLEBA2,
            'TEM_INTERSECAO_CAR_SUSPENSO',
            'QUANTIDADE_CADASTROS_CAR_SUSPENSOS_INTERSECTADOS',
            'AREA_INTERSECAO_TOTAL_HA',
            'PERCENTUAL_MAXIMO_GLEBA_INTERSECTADA',
        ]
    ]

resumo_por_id.to_csv(
    ARQUIVO_RESUMO_POR_ID,
    index=False,
    sep=SEPARADOR_SAIDA,
    decimal=',',
    encoding='utf-8-sig',
)

print('\nResultado único salvo em:')
print(ARQUIVO_INTERSECAO_POR_ID)
print('\nResumo de todos os IDs salvo em:')
print(ARQUIVO_RESUMO_POR_ID)
print('\nIDs com interseção:', f"{resumo_por_id['TEM_INTERSECAO_CAR_SUSPENSO'].sum():,}")
print('IDs sem interseção:', f"{(~resumo_por_id['TEM_INTERSECAO_CAR_SUSPENSO']).sum():,}")


In [ ]:
# ============================================================
# 8. DEVOLVER A RESPOSTA A TODAS AS LINHAS DO SICOR
# ============================================================

# A interseção não é recalculada aqui. O merge apenas replica a resposta
# de cada ID_GLEBA2 para todos os REF_BACEN, ordens, índices e anos ligados a ele.

colunas_espaciais_para_merge = [
    coluna
    for coluna in intersecoes_por_id.columns
    if coluna != COLUNA_ID_GLEBA2
]

escrever_cabecalho = True
total_linhas_originais_reprocessadas = 0
total_linhas_expandidas = 0

leitor_expandir = pd.read_csv(
    ARQUIVO_SICOR,
    sep=separador_sicor,
    encoding=encoding_sicor,
    dtype=str,
    chunksize=TAMANHO_BLOCO,
    low_memory=False,
)

for numero_bloco, bloco in enumerate(leitor_expandir, start=1):
    bloco = normalizar_colunas(bloco)
    bloco[COLUNA_ID_GLEBA2] = padronizar_codigo(bloco[COLUNA_ID_GLEBA2])
    total_linhas_originais_reprocessadas += len(bloco)

    if intersecoes_por_id.empty:
        expandido = pd.DataFrame()
    else:
        expandido = bloco.merge(
            intersecoes_por_id,
            on=COLUNA_ID_GLEBA2,
            how='inner',
            validate='many_to_many',
            suffixes=('', '_INTERSECAO'),
        )

    if expandido.empty:
        print(f'Bloco {numero_bloco:03d}: nenhuma linha com interseção')
        del bloco, expandido
        gc.collect()
        continue

    expandido.to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        mode='w' if escrever_cabecalho else 'a',
        header=escrever_cabecalho,
        index=False,
        sep=SEPARADOR_SAIDA,
        decimal=',',
        encoding='utf-8-sig' if escrever_cabecalho else 'utf-8',
    )

    total_linhas_expandidas += len(expandido)
    escrever_cabecalho = False

    print(
        f'Bloco {numero_bloco:03d}: originais={len(bloco):,} | '
        f'linhas de interseção expandidas={len(expandido):,}'
    )

    del bloco, expandido
    gc.collect()

if escrever_cabecalho:
    # Nenhuma interseção: ainda cria um CSV vazio com cabeçalho conhecido.
    pd.DataFrame(
        columns=[*colunas_sicor, *colunas_espaciais_para_merge]
    ).to_csv(
        ARQUIVO_INTERSECAO_EXPANDIDA,
        index=False,
        sep=SEPARADOR_SAIDA,
        encoding='utf-8-sig',
    )

if total_linhas_originais_reprocessadas != total_linhas_lidas:
    raise RuntimeError(
        'A quantidade de linhas relidas para a expansão difere da '
        'quantidade lida na etapa espacial.'
    )

print('\nRESULTADO DETALHADO EXPANDIDO')
print('=' * 90)
print('Linhas originais relidas:', f'{total_linhas_originais_reprocessadas:,}')
print('Linhas finais de interseção:', f'{total_linhas_expandidas:,}')
print('Arquivo salvo em:')
print(ARQUIVO_INTERSECAO_EXPANDIDA)


In [ ]:
# ============================================================
# 9. CONFERÊNCIA FINAL
# ============================================================

print('ARQUIVOS GERADOS')
print('=' * 100)

for caminho in [
    ARQUIVO_INTERSECAO_POR_ID,
    ARQUIVO_INTERSECAO_EXPANDIDA,
    ARQUIVO_RESUMO_POR_ID,
    ARQUIVO_DIAGNOSTICO_STATUS,
]:
    print(f"{'OK' if caminho.exists() else 'NÃO CRIADO'} | {caminho}")

print('\nAMOSTRA DO RESULTADO ÚNICO POR ID_GLEBA2')
display(intersecoes_por_id.head(5))

print('\nAMOSTRA DO RESUMO, INCLUSIVE IDs SEM INTERSEÇÃO')
display(resumo_por_id.head(5))

print('\nESTATÍSTICAS')
print('ID_GLEBA2 distintos analisados:', f'{len(resumo_por_id):,}')
print('ID_GLEBA2 com interseção:', f"{resumo_por_id['TEM_INTERSECAO_CAR_SUSPENSO'].sum():,}")
print('Pares ID_GLEBA2 × cadastro do SICAR suspenso:', f'{len(intersecoes_por_id):,}')

if not intersecoes_por_id.empty:
    print(
        'Área total das interseções únicas (ha):',
        f"{intersecoes_por_id['AREA_INTERSECAO_HA'].sum():,.4f}",
    )
